In [1]:
#!/usr/bin/env python3.11

# Copyright 2025, Gurobi Optimization, LLC

# Sudoku example.

# The Sudoku board is a 9x9 grid, which is further divided into a 3x3 grid
# of 3x3 grids.  Each cell in the grid must take a value from 0 to 9.
# No two grid cells in the same row, column, or 3x3 subgrid may take the
# same value.
#
# In the MIP formulation, binary variables x[i,j,v] indicate whether
# cell <i,j> takes value 'v'.  The constraints are as follows:
#   1. Each cell must take exactly one value (sum_v x[i,j,v] = 1)
#   2. Each value is used exactly once per row (sum_i x[i,j,v] = 1)
#   3. Each value is used exactly once per column (sum_j x[i,j,v] = 1)
#   4. Each value is used exactly once per 3x3 subgrid (sum_grid x[i,j,v] = 1)
#
# Input datasets for this example can be found in examples/data/sudoku*.

import math
import gurobipy as gp
from gurobipy import GRB

f = open("sudoku.txt")

grid = f.read().split()

n = len(grid[0])
s = int(math.sqrt(n))

# Create our 3-D array of model variables

model = gp.Model("sudoku")

vars = model.addVars(n, n, n, vtype=GRB.BINARY, name="G")

# Fix variables associated with cells whose values are pre-specified

for i in range(n):
    for j in range(n):
        if grid[i][j] != ".":
            v = int(grid[i][j]) - 1
            vars[i, j, v].LB = 1

model.addConstrs(
    (gp.quicksum(vars[i,j,v] for i in range(n)) == 1 for j in range(n) for v in range(n)), name="V"
)
model.addConstrs(
    (gp.quicksum(vars[i,j,v] for j in range(n)) == 1 for i in range(n) for v in range(n)), name="V"
)
model.addConstrs(
    (gp.quicksum(vars[i,j,v] for v in range(n)) == 1 for i in range(n) for j in range(n)), name="V"
)

# Each cell must take one value

# model.addConstrs(
#     (vars.sum(i, j, "*") == 1 for i in range(n) for j in range(n)), name="V"
# )

# Each value appears once per row

# model.addConstrs(
#     (vars.sum(i, "*", v) == 1 for i in range(n) for v in range(n)), name="R"
# )

# Each value appears once per column

# model.addConstrs(
#     (vars.sum("*", j, v) == 1 for j in range(n) for v in range(n)), name="C"
# )


# Each value appears once per subgrid

model.addConstrs(
    (
        gp.quicksum(
            vars[i, j, v]
            for i in range(i0 * s, (i0 + 1) * s)
            for j in range(j0 * s, (j0 + 1) * s)
        )
        == 1
        for v in range(n)
        for i0 in range(s)
        for j0 in range(s)
    ),
    name="Sub",
)


model.optimize()
print("Constrains: ", model.NumConstrs)

model.write("sudoku.lp")

# Mostrar el grid del Sudoku
print("\nSolución del Sudoku:\n")

solution = model.getAttr("X", vars)
for i in range(n):
    row = []
    for j in range(n):
        for v in range(n):
            if solution[i, j, v] > 0.5:
                row.append(str(v + 1))  # Convertir el valor a string

    # Imprimir la fila con separadores de cuadrantes
    print("| "+" | ".join([" ".join(row[j:j+s]) for j in range(0, n, s)]) + " |")  # Separar por subgrids

    # Agregar una línea horizontal cada s filas (excepto al final)
    if (i + 1) % s == 0 and i + 1 < n:
        print("-" * ((n + s)*2+1))  # Línea horizontal con separadores

Set parameter Username


Academic license - for non-commercial use only - expires 2026-06-16
Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (win64 - Windows 10.0 (19045.2))

CPU model: 11th Gen Intel(R) Core(TM) i7-11800H @ 2.30GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 324 rows, 729 columns and 2916 nonzeros
Model fingerprint: 0xa5388ad7
Variable types: 0 continuous, 729 integer (729 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [0e+00, 0e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]
Presolve removed 324 rows and 729 columns
Presolve time: 0.02s
Presolve: All rows and columns removed

Explored 0 nodes (0 simplex iterations) in 0.02 seconds (0.00 work units)
Thread count was 1 (of 16 available processors)

Solution count 1: 0 

Optimal solution found (tolerance 1.00e-04)
Best objective 0.000000000000e+00, best bound 0.000000000000e+00, gap 0.0